## Langfuse Cloud Connection

Initialize the Langfuse client and verify that the application can connect to the cloud observability platform.

In [1]:
import os
from dotenv import load_dotenv
from langfuse import get_client

load_dotenv()

langfuse = get_client()

print("Public key configured:", bool(os.getenv("LANGFUSE_PUBLIC_KEY")))
print("Secret key configured:", bool(os.getenv("LANGFUSE_SECRET_KEY")))
print("Authenticated:", langfuse.auth_check())

Public key configured: True
Secret key configured: True
Authenticated: True


## 1. Inspect Existing Vector Indexes

Inspect the Discovery and Detail Chroma collections to understand their document counts, collection configuration, IDs and metadata structure.

This establishes the baseline before implementing indexing CRUD operations.

In [2]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

GOLDEN_PATH = (
    PROJECT_DIR
    / "notebooks/golden_dataset/discovery/seva_discovery_golden_v1.jsonl"
)

DISCOVERY_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\discovery")

DETAIL_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\detail")

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [3]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 182.63 seconds
Embedding dimensions: 384
Model loaded successfully!


In [4]:
from collections import defaultdict
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

VECTORSTORE_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores")
golden_df = pd.read_json(GOLDEN_PATH, lines=True)


discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [5]:
import time
from langfuse import get_client

langfuse = get_client()

with langfuse.start_as_current_observation(
    name="indexing.inspect",
    as_type="span"
) as span:

    start = time.perf_counter()

    discovery_collection = discovery_db._collection
    detail_collection = detail_db._collection

    discovery_count = discovery_collection.count()
    detail_count = detail_collection.count()

    discovery_sample = discovery_collection.get(
        limit=3,
        include=["metadatas", "documents"]
    )

    detail_sample = detail_collection.get(
        limit=3,
        include=["metadatas", "documents"]
    )

    latency = time.perf_counter() - start

    span.update(
        metadata={
            "discovery_count": discovery_count,
            "detail_count": detail_count,
            "sample_size": 3
        },
        level="DEFAULT"
    )

print("DISCOVERY COLLECTION")
print("Name:", discovery_collection.name)
print("Count:", discovery_count)
print("Metadata:", discovery_collection.metadata)
print("Sample IDs:", discovery_sample["ids"])
print("Sample metadata:", discovery_sample["metadatas"])

print("\nDETAIL COLLECTION")
print("Name:", detail_collection.name)
print("Count:", detail_count)
print("Metadata:", detail_collection.metadata)
print("Sample IDs:", detail_sample["ids"])
print("Sample metadata:", detail_sample["metadatas"])

print(f"\nInspection latency: {latency:.4f} seconds")

langfuse.flush()

DISCOVERY COLLECTION
Name: seva_discovery_v1
Count: 23438
Metadata: None
Sample IDs: ['LBezF5gB5uF80_MGGY81_OVERVIEW_001_CHUNK_001', 'LBezF5gB5uF80_MGGY81_BENEFITS_001_CHUNK_001', 'LBezF5gB5uF80_MGGY81_BENEFICIARIES_001_CHUNK_001']
Sample metadata: [{'parent_id': 'LBezF5gB5uF80_MGGY81_OVERVIEW_001', 'level': 'Central', 'chunk_id': 'LBezF5gB5uF80_MGGY81_OVERVIEW_001_CHUNK_001', 'child_type': 'overview', 'scheme_id': 'LBezF5gB5uF80_MGGY81', 'categories': '["Business & Entrepreneurship", "Banking,Financial Services and Insurance", "Social welfare & Empowerment"]', 'source_field': 'brief_description', 'chunk_index': 1, 'child_id': 'LBezF5gB5uF80_MGGY81_OVERVIEW_001', 'language': 'en', 'states': '["All"]'}, {'level': 'Central', 'chunk_id': 'LBezF5gB5uF80_MGGY81_BENEFITS_001_CHUNK_001', 'scheme_id': 'LBezF5gB5uF80_MGGY81', 'categories': '["Business & Entrepreneurship", "Banking,Financial Services and Insurance", "Social welfare & Empowerment"]', 'parent_id': 'LBezF5gB5uF80_MGGY81_BENEFITS_00

## 2. Validate Document IDs and Relationships

Analyze the metadata of both Chroma collections to validate vector IDs, scheme IDs, parent-child relationships and chunk distribution.

This helps establish safe identifiers for insertion, update and deletion operations.

In [7]:
import pandas as pd
import time

BATCH_SIZE = 500

def load_collection_metadata(collection):
    records = []
    total = collection.count()

    for offset in range(0, total, BATCH_SIZE):
        batch = collection.get(
            limit=BATCH_SIZE,
            offset=offset,
            include=["metadatas"]
        )

        for vector_id, metadata in zip(
            batch["ids"], batch["metadatas"]
        ):
            records.append({
                "vector_id": vector_id,
                **(metadata or {})
            })

    return pd.DataFrame(records)


with langfuse.start_as_current_observation(
    name="indexing.validate_relationships",
    as_type="span"
) as span:

    start = time.perf_counter()

    collection_data = {
        "discovery": load_collection_metadata(discovery_collection),
        "detail": load_collection_metadata(detail_collection)
    }

    required = ["scheme_id", "parent_id", "child_id", "chunk_id"]

    for name, df in collection_data.items():

        for field in required:
            if field not in df.columns:
                df[field] = None

        df["id_matches_chunk_id"] = (
            df["vector_id"] == df["chunk_id"]
        )

        print(f"\n{name.upper()}")
        print("Total chunks:", len(df))
        print("Unique schemes:", df["scheme_id"].nunique())
        print("Unique parents:", df["parent_id"].nunique())
        print("Unique children:", df["child_id"].nunique())

        print("Missing metadata:")
        print(df[required].isna().sum().to_dict())

        print(
            "Vector ID mismatches:",
            (~df["id_matches_chunk_id"]).sum()
        )

        print("\nChunks per child:")
        print(df.groupby("child_id").size().describe().round(2))

    discovery_schemes = set(
        collection_data["discovery"]["scheme_id"].dropna()
    )

    detail_schemes = set(
        collection_data["detail"]["scheme_id"].dropna()
    )

    overlap = discovery_schemes & detail_schemes

    latency = time.perf_counter() - start

    span.update(
        metadata={
            "discovery_schemes": len(discovery_schemes),
            "detail_schemes": len(detail_schemes),
            "shared_schemes": len(overlap),
            "batch_size": BATCH_SIZE
        }
    )

print("\nCROSS-COLLECTION ANALYSIS")
print("Discovery schemes:", len(discovery_schemes))
print("Detail schemes:", len(detail_schemes))
print("Shared schemes:", len(overlap))
print(f"\nValidation latency: {latency:.4f} seconds")

langfuse.flush()


DISCOVERY
Total chunks: 23438
Unique schemes: 4428
Unique parents: 18393
Unique children: 18393
Missing metadata:
{'scheme_id': 0, 'parent_id': 0, 'child_id': 0, 'chunk_id': 0}
Vector ID mismatches: 0

Chunks per child:
count    18393.00
mean         1.27
std          0.98
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         42.00
dtype: float64

DETAIL
Total chunks: 67945
Unique schemes: 4426
Unique parents: 57973
Unique children: 57973
Missing metadata:
{'scheme_id': 0, 'parent_id': 0, 'child_id': 0, 'chunk_id': 0}
Vector ID mismatches: 0

Chunks per child:
count    57973.00
mean         1.17
std          0.74
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         17.00
dtype: float64

CROSS-COLLECTION ANALYSIS
Discovery schemes: 4428
Detail schemes: 4426
Shared schemes: 4426

Validation latency: 114.7646 seconds


## 3. Investigate Discovery-Only Schemes

Identify schemes present in Discovery but absent from Detail. Inspect their available sections, parent IDs and chunk counts to determine whether they are intentionally overview-only or have incomplete detail indexing.

In [8]:
import time
import pandas as pd

with langfuse.start_as_current_observation(
    name="indexing.investigate_missing_schemes",
    as_type="span"
) as span:

    start = time.perf_counter()

    discovery_df = collection_data["discovery"]
    detail_df = collection_data["detail"]

    discovery_only = sorted(
        set(discovery_df["scheme_id"].dropna())
        - set(detail_df["scheme_id"].dropna())
    )

    missing_scheme_df = discovery_df[
        discovery_df["scheme_id"].isin(discovery_only)
    ].copy()

    summary = (
        missing_scheme_df
        .groupby("scheme_id")
        .agg(
            chunks=("vector_id", "count"),
            parents=("parent_id", "nunique"),
            sections=("child_type", lambda x: sorted(x.dropna().unique().tolist())),
            source_fields=("source_field", lambda x: sorted(x.dropna().unique().tolist()))
        )
        .reset_index()
    )

    latency = time.perf_counter() - start

    span.update(
        metadata={
            "discovery_only_scheme_count": len(discovery_only),
            "affected_chunk_count": len(missing_scheme_df)
        }
    )

print("Discovery-only scheme IDs:")
print(discovery_only)

print("\nScheme-level investigation:")
display(summary)

print(f"\nInvestigation latency: {latency:.4f} seconds")

langfuse.flush()

Discovery-only scheme IDs:
['5aKstZ4BXu-iUKXq-liy', 'DRi4Yp0B5uF80_MGMmWm']

Scheme-level investigation:


,scheme_id,chunks,parents,sections,source_fields
0,5aKstZ4BXu-iUKXq-liy,1,1,[overview],[brief_description]
1,DRi4Yp0B5uF80_MGMmWm,1,1,[overview],[brief_description]



Investigation latency: 1.0189 seconds


## 4. Inspect Embedding and Collection Configuration

Inspect the existing embedding function, vector dimensions and collection configuration. New documents must use the same embedding dimensions and compatible distance metric as the existing collection.

In [9]:
import time

with langfuse.start_as_current_observation(
    name="indexing.inspect_configuration",
    as_type="span"
) as span:

    start = time.perf_counter()

    for name, collection, vectorstore in [
        ("discovery", discovery_collection, discovery_db),
        ("detail", detail_collection, detail_db)
    ]:
        sample = collection.get(
            limit=1,
            include=["embeddings"]
        )

        embedding_dimension = len(sample["embeddings"][0])

        print(f"\n{name.upper()}")
        print("Embedding dimension:", embedding_dimension)
        print("Embedding function:", type(
            getattr(vectorstore, "_embedding_function", None)
        ).__name__)
        print("Collection configuration:", getattr(
            collection, "configuration", None
        ))

    latency = time.perf_counter() - start

    span.update(metadata={
        "inspection_latency_seconds": latency
    })

print(f"\nConfiguration inspection latency: {latency:.4f} seconds")

langfuse.flush()


DISCOVERY
Embedding dimension: 384
Embedding function: HuggingFaceEmbeddings
Collection configuration: {'hnsw': {'space': 'l2', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': None}

DETAIL
Embedding dimension: 384
Embedding function: HuggingFaceEmbeddings
Collection configuration: {'hnsw': {'space': 'l2', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': None}

Configuration inspection latency: 0.0969 seconds


## 5. Create an Isolated CRUD Test Collection

Create a temporary Chroma collection using the existing HuggingFace embedding model and L2 distance metric.

Use this collection to test insertion, reading, updating and deletion before applying these operations to the actual SEVA_AI indexes.

In [10]:
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document

crud_client = chromadb.EphemeralClient()

crud_db = Chroma(
    client=crud_client,
    collection_name="seva_crud_test_v1",
    embedding_function=discovery_db._embedding_function,
    collection_metadata={"hnsw:space": "l2"}
)

print("Collection:", crud_db._collection.name)
print("Initial count:", crud_db._collection.count())
print("Embedding model:", type(crud_db._embedding_function).__name__)
print("Distance metric:", crud_db._collection.configuration["hnsw"]["space"])

Collection: seva_crud_test_v1
Initial count: 0
Embedding model: HuggingFaceEmbeddings
Distance metric: l2


## 6. Insert a Test Document

Insert one test scheme document into the temporary collection. Capture the operation in Langfuse to automatically record execution duration and indexing metadata.

In [11]:
test_id = "CRUD_TEST_SCHEME_001_OVERVIEW_CHUNK_001"

test_doc = Document(
    page_content="This is a test government scheme for validating SEVA_AI indexing CRUD operations.",
    metadata={
        "scheme_id": "CRUD_TEST_SCHEME_001",
        "parent_id": "CRUD_TEST_SCHEME_001_OVERVIEW_001",
        "child_id": "CRUD_TEST_SCHEME_001_OVERVIEW_001",
        "chunk_id": test_id,
        "child_type": "overview",
        "source_field": "brief_description",
        "language": "en",
        "level": "Central"
    }
)

with langfuse.start_as_current_observation(
    name="indexing.insert",
    as_type="span"
) as span:

    inserted_ids = crud_db.add_documents(
        documents=[test_doc],
        ids=[test_id]
    )

    span.update(metadata={
        "operation": "insert",
        "collection": "seva_crud_test_v1",
        "documents_inserted": len(inserted_ids),
        "batch_size": 1
    })

print("Inserted IDs:", inserted_ids)
print("Collection count:", crud_db._collection.count())

langfuse.flush()

Inserted IDs: ['CRUD_TEST_SCHEME_001_OVERVIEW_CHUNK_001']
Collection count: 1


## 7. Read an Indexed Document

Retrieve the inserted document using its vector ID. Validate its content, metadata and vector ID against the original record.

In [12]:
with langfuse.start_as_current_observation(
    name="indexing.read",
    as_type="span"
) as span:

    result = crud_db._collection.get(
        ids=[test_id],
        include=["documents", "metadatas"]
    )

    span.update(metadata={
        "operation": "read",
        "collection": "seva_crud_test_v1",
        "records_requested": 1,
        "records_returned": len(result["ids"])
    })

print("Retrieved ID:", result["ids"][0])
print("Content:", result["documents"][0])
print("Metadata:", result["metadatas"][0])

print("\nID matches:", result["ids"][0] == test_id)
print("Content matches:", result["documents"][0] == test_doc.page_content)
print("Metadata matches:", result["metadatas"][0] == test_doc.metadata)

langfuse.flush()

Retrieved ID: CRUD_TEST_SCHEME_001_OVERVIEW_CHUNK_001
Content: This is a test government scheme for validating SEVA_AI indexing CRUD operations.
Metadata: {'language': 'en', 'child_id': 'CRUD_TEST_SCHEME_001_OVERVIEW_001', 'chunk_id': 'CRUD_TEST_SCHEME_001_OVERVIEW_CHUNK_001', 'parent_id': 'CRUD_TEST_SCHEME_001_OVERVIEW_001', 'child_type': 'overview', 'scheme_id': 'CRUD_TEST_SCHEME_001', 'source_field': 'brief_description', 'level': 'Central'}

ID matches: True
Content matches: True
Metadata matches: True


## 8. Update an Indexed Document

Update the existing document using its vector ID. Modify its content and metadata, regenerate its embedding, and verify that the original vector ID remains unchanged.

In [13]:
import numpy as np
from langchain_core.documents import Document

updated_doc = Document(
    page_content="This updated government scheme is used to test SEVA_AI document updates and embedding regeneration.",
    metadata={
        **test_doc.metadata,
        "status": "updated",
        "version": 2
    }
)

with langfuse.start_as_current_observation(
    name="indexing.update",
    as_type="span"
) as span:

    old_embedding = crud_db._collection.get(
        ids=[test_id],
        include=["embeddings"]
    )["embeddings"][0]

    crud_db.update_documents(
        ids=[test_id],
        documents=[updated_doc]
    )

    updated_result = crud_db._collection.get(
        ids=[test_id],
        include=["documents", "metadatas", "embeddings"]
    )

    new_embedding = updated_result["embeddings"][0]

    embedding_changed = not np.allclose(
        old_embedding, new_embedding
    )

    span.update(metadata={
        "operation": "update",
        "collection": "seva_crud_test_v1",
        "documents_updated": 1,
        "embedding_regenerated": embedding_changed
    })

print("Updated ID:", updated_result["ids"][0])
print("Updated content:", updated_result["documents"][0])
print("Updated metadata:", updated_result["metadatas"][0])
print("Embedding regenerated:", embedding_changed)
print("Collection count:", crud_db._collection.count())
print("ID preserved:", updated_result["ids"][0] == test_id)

langfuse.flush()

Updated ID: CRUD_TEST_SCHEME_001_OVERVIEW_CHUNK_001
Updated content: This updated government scheme is used to test SEVA_AI document updates and embedding regeneration.
Updated metadata: {'scheme_id': 'CRUD_TEST_SCHEME_001', 'status': 'updated', 'source_field': 'brief_description', 'version': 2, 'language': 'en', 'level': 'Central', 'child_type': 'overview', 'chunk_id': 'CRUD_TEST_SCHEME_001_OVERVIEW_CHUNK_001', 'child_id': 'CRUD_TEST_SCHEME_001_OVERVIEW_001', 'parent_id': 'CRUD_TEST_SCHEME_001_OVERVIEW_001'}
Embedding regenerated: True
Collection count: 1
ID preserved: True


## 9. Delete an Indexed Document

Delete the test document using its vector ID. Verify that the record is removed and the collection count returns to zero.

In [14]:
with langfuse.start_as_current_observation(
    name="indexing.delete",
    as_type="span"
) as span:

    before_count = crud_db._collection.count()

    crud_db.delete(ids=[test_id])

    after_count = crud_db._collection.count()

    deleted_record = crud_db._collection.get(
        ids=[test_id],
        include=["metadatas"]
    )

    deletion_verified = len(deleted_record["ids"]) == 0

    span.update(metadata={
        "operation": "delete",
        "collection": "seva_crud_test_v1",
        "records_deleted": before_count - after_count,
        "deletion_verified": deletion_verified
    })

print("Before deletion:", before_count)
print("After deletion:", after_count)
print("Record exists:", not deletion_verified)
print("Deletion verified:", deletion_verified)

langfuse.flush()

Before deletion: 1
After deletion: 0
Record exists: False
Deletion verified: True


## 10. Scheme-Level Read

Retrieve all indexed chunks belonging to a specific scheme from both Discovery and Detail collections using the scheme_id metadata filter.

Measure operation latency and inspect the number of chunks and sections returned.

In [15]:
test_scheme_id = "LBezF5gB5uF80_MGGY81"

with langfuse.start_as_current_observation(
    name="indexing.read_scheme",
    as_type="span"
) as span:

    start = time.perf_counter()

    discovery_records = discovery_collection.get(
        where={"scheme_id": test_scheme_id},
        include=["metadatas"]
    )

    detail_records = detail_collection.get(
        where={"scheme_id": test_scheme_id},
        include=["metadatas"]
    )

    latency = time.perf_counter() - start

    span.update(metadata={
        "scheme_id": test_scheme_id,
        "discovery_chunks": len(discovery_records["ids"]),
        "detail_chunks": len(detail_records["ids"])
    })

print("Scheme ID:", test_scheme_id)

print("\nDiscovery")
print("Chunks:", len(discovery_records["ids"]))
print("Sections:", sorted({
    m.get("child_type")
    for m in discovery_records["metadatas"]
    if m.get("child_type")
}))

print("\nDetail")
print("Chunks:", len(detail_records["ids"]))
print("Sections:", sorted({
    m.get("child_type")
    for m in detail_records["metadatas"]
    if m.get("child_type")
}))

print("\nTotal chunks:",
      len(discovery_records["ids"]) + len(detail_records["ids"]))

print(f"Scheme read latency: {latency:.4f} seconds")

langfuse.flush()

Scheme ID: LBezF5gB5uF80_MGGY81

Discovery
Chunks: 4
Sections: ['beneficiaries', 'benefits', 'eligibility', 'overview']

Detail
Chunks: 17
Sections: ['application_process', 'documents_required', 'faq']

Total chunks: 21
Scheme read latency: 0.2152 seconds


## 11. Scheme-Level Insert

Create two isolated Chroma collections representing Discovery and Detail indexes. Insert a synthetic scheme containing multiple parent documents and child chunks across both collections.

Use stable chunk IDs and scheme-level metadata. Capture the complete operation and individual collection insertion spans in Langfuse.

In [16]:
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Create fresh isolated test collections
crud_client = chromadb.EphemeralClient()

crud_discovery_db = Chroma(
    client=crud_client,
    collection_name="seva_discovery_crud_test_v1",
    embedding_function=discovery_db._embedding_function,
    collection_metadata={"hnsw:space": "l2"}
)

crud_detail_db = Chroma(
    client=crud_client,
    collection_name="seva_detail_crud_test_v1",
    embedding_function=detail_db._embedding_function,
    collection_metadata={"hnsw:space": "l2"}
)

scheme_id = "CRUD_TEST_SCHEME_002"

discovery_docs = [
    Document(
        page_content="This synthetic scheme provides financial assistance to eligible entrepreneurs.",
        metadata={
            "scheme_id": scheme_id,
            "parent_id": f"{scheme_id}_OVERVIEW_001",
            "child_id": f"{scheme_id}_OVERVIEW_001",
            "chunk_id": f"{scheme_id}_OVERVIEW_001_CHUNK_001",
            "child_type": "overview",
            "source_field": "brief_description",
            "chunk_index": 1,
            "language": "en",
            "level": "Central"
        }
    ),
    Document(
        page_content="Eligible beneficiaries can receive financial support under this scheme.",
        metadata={
            "scheme_id": scheme_id,
            "parent_id": f"{scheme_id}_BENEFITS_001",
            "child_id": f"{scheme_id}_BENEFITS_001",
            "chunk_id": f"{scheme_id}_BENEFITS_001_CHUNK_001",
            "child_type": "benefits",
            "source_field": "benefits",
            "chunk_index": 1,
            "language": "en",
            "level": "Central"
        }
    )
]

detail_docs = [
    Document(
        page_content="Applicants must submit their application through the official portal.",
        metadata={
            "scheme_id": scheme_id,
            "parent_id": f"{scheme_id}_APPLICATION_001",
            "child_id": f"{scheme_id}_APPLICATION_001",
            "chunk_id": f"{scheme_id}_APPLICATION_001_CHUNK_001",
            "child_type": "application_process",
            "source_field": "application_process",
            "chunk_index": 1,
            "language": "en",
            "level": "Central"
        }
    ),
    Document(
        page_content="Applicants must provide identity proof and address proof.",
        metadata={
            "scheme_id": scheme_id,
            "parent_id": f"{scheme_id}_FAQ_001",
            "child_id": f"{scheme_id}_FAQ_001",
            "chunk_id": f"{scheme_id}_FAQ_001_CHUNK_001",
            "child_type": "faq",
            "source_field": "faq",
            "chunk_index": 1,
            "language": "en",
            "level": "Central"
        }
    ),
    Document(
        page_content="Applicants must also provide their bank account details.",
        metadata={
            "scheme_id": scheme_id,
            "parent_id": f"{scheme_id}_FAQ_001",
            "child_id": f"{scheme_id}_FAQ_001",
            "chunk_id": f"{scheme_id}_FAQ_001_CHUNK_002",
            "child_type": "faq",
            "source_field": "faq",
            "chunk_index": 2,
            "language": "en",
            "level": "Central"
        }
    )
]

with langfuse.start_as_current_observation(
    name="indexing.insert_scheme",
    as_type="span"
) as span:

    with langfuse.start_as_current_observation(
        name="indexing.insert_scheme.discovery",
        as_type="span"
    ) as discovery_span:

        discovery_ids = crud_discovery_db.add_documents(
            documents=discovery_docs,
            ids=[doc.metadata["chunk_id"] for doc in discovery_docs]
        )

        discovery_span.update(metadata={
            "scheme_id": scheme_id,
            "documents_inserted": len(discovery_ids)
        })

    with langfuse.start_as_current_observation(
        name="indexing.insert_scheme.detail",
        as_type="span"
    ) as detail_span:

        detail_ids = crud_detail_db.add_documents(
            documents=detail_docs,
            ids=[doc.metadata["chunk_id"] for doc in detail_docs]
        )

        detail_span.update(metadata={
            "scheme_id": scheme_id,
            "documents_inserted": len(detail_ids)
        })

    span.update(metadata={
        "scheme_id": scheme_id,
        "discovery_chunks": len(discovery_ids),
        "detail_chunks": len(detail_ids),
        "total_chunks": len(discovery_ids) + len(detail_ids)
    })

print("Scheme ID:", scheme_id)
print("Discovery inserted:", len(discovery_ids))
print("Detail inserted:", len(detail_ids))
print("Total inserted:", len(discovery_ids) + len(detail_ids))

print("\nCollection counts:")
print("Discovery:", crud_discovery_db._collection.count())
print("Detail:", crud_detail_db._collection.count())

langfuse.flush()

Scheme ID: CRUD_TEST_SCHEME_002
Discovery inserted: 2
Detail inserted: 3
Total inserted: 5

Collection counts:
Discovery: 2
Detail: 3


## 12. Verify Scheme-Level Read

Retrieve all documents belonging to the synthetic scheme from Discovery and Detail collections. Validate chunk IDs, parent-child relationships, metadata and document counts.

In [17]:
with langfuse.start_as_current_observation(
    name="indexing.read_scheme",
    as_type="span"
) as span:

    discovery_result = crud_discovery_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["documents", "metadatas"]
    )

    detail_result = crud_detail_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["documents", "metadatas"]
    )

    discovery_count = len(discovery_result["ids"])
    detail_count = len(detail_result["ids"])

    all_ids = discovery_result["ids"] + detail_result["ids"]

    faq_chunks = [
        metadata
        for metadata in detail_result["metadatas"]
        if metadata.get("child_type") == "faq"
    ]

    faq_parent_ids = {
        metadata["parent_id"] for metadata in faq_chunks
    }

    span.update(metadata={
        "scheme_id": scheme_id,
        "discovery_chunks": discovery_count,
        "detail_chunks": detail_count,
        "total_chunks": len(all_ids)
    })

print("Scheme ID:", scheme_id)
print("Discovery chunks:", discovery_count)
print("Detail chunks:", detail_count)
print("Total chunks:", len(all_ids))

print("\nUnique vector IDs:", len(set(all_ids)))
print("FAQ chunks:", len(faq_chunks))
print("FAQ parent count:", len(faq_parent_ids))

print("\nExpected counts verified:",
      discovery_count == 2 and detail_count == 3)

print("All vector IDs unique:", len(all_ids) == len(set(all_ids)))
print("FAQ parent relationship preserved:", len(faq_parent_ids) == 1)

langfuse.flush()

Scheme ID: CRUD_TEST_SCHEME_002
Discovery chunks: 2
Detail chunks: 3
Total chunks: 5

Unique vector IDs: 5
FAQ chunks: 2
FAQ parent count: 1

Expected counts verified: True
All vector IDs unique: True
FAQ parent relationship preserved: True


## 13. Scheme-Level Update

Update all documents belonging to a scheme across Discovery and Detail collections. Regenerate embeddings for modified content while preserving stable vector IDs and parent-child relationships.

In [18]:
import numpy as np

updated_discovery_docs = [
    Document(
        page_content=doc.page_content + " Updated for CRUD version two.",
        metadata={**doc.metadata, "version": 2, "status": "updated"}
    )
    for doc in discovery_docs
]

updated_detail_docs = [
    Document(
        page_content=doc.page_content + " Updated for CRUD version two.",
        metadata={**doc.metadata, "version": 2, "status": "updated"}
    )
    for doc in detail_docs
]

with langfuse.start_as_current_observation(
    name="indexing.update_scheme",
    as_type="span"
) as span:

    start = time.perf_counter()

    # Capture old embeddings
    old_discovery = crud_discovery_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["embeddings"]
    )

    old_detail = crud_detail_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["embeddings"]
    )

    # Update both collections
    crud_discovery_db.update_documents(
        ids=discovery_ids,
        documents=updated_discovery_docs
    )

    crud_detail_db.update_documents(
        ids=detail_ids,
        documents=updated_detail_docs
    )

    # Verify updated records
    new_discovery = crud_discovery_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["documents", "metadatas", "embeddings"]
    )

    new_detail = crud_detail_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["documents", "metadatas", "embeddings"]
    )

    old_embeddings = (
        list(old_discovery["embeddings"])
        + list(old_detail["embeddings"])
    )

    new_embeddings = (
        list(new_discovery["embeddings"])
        + list(new_detail["embeddings"])
    )

    embedding_changes = [
        not np.allclose(old, new)
        for old, new in zip(old_embeddings, new_embeddings)
    ]

    latency = time.perf_counter() - start

    span.update(metadata={
        "scheme_id": scheme_id,
        "documents_updated": len(discovery_ids) + len(detail_ids),
        "embeddings_changed": sum(embedding_changes),
        "latency_seconds": latency
    })

print("Discovery updated:", len(new_discovery["ids"]))
print("Detail updated:", len(new_detail["ids"]))
print("Total updated:", len(new_discovery["ids"]) + len(new_detail["ids"]))

print("Embeddings changed:", sum(embedding_changes))
print("All metadata version 2:", all(
    m.get("version") == 2
    for m in new_discovery["metadatas"] + new_detail["metadatas"]
))

print("All statuses updated:", all(
    m.get("status") == "updated"
    for m in new_discovery["metadatas"] + new_detail["metadatas"]
))

print("Vector IDs preserved:",
      set(new_discovery["ids"]) == set(discovery_ids)
      and set(new_detail["ids"]) == set(detail_ids))

print(f"Total update latency: {latency:.4f} seconds")

langfuse.flush()

Discovery updated: 2
Detail updated: 3
Total updated: 5
Embeddings changed: 5
All metadata version 2: True
All statuses updated: True
Vector IDs preserved: True
Total update latency: 0.2908 seconds


## 14. Scheme-Level Delete

Delete all documents belonging to a scheme from both Discovery and Detail collections using metadata filtering. Verify that all associated chunks are removed and unrelated records remain untouched.

In [19]:
with langfuse.start_as_current_observation(
    name="indexing.delete_scheme",
    as_type="span"
) as span:

    start = time.perf_counter()

    discovery_before = crud_discovery_db._collection.count()
    detail_before = crud_detail_db._collection.count()

    discovery_db_records = crud_discovery_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["metadatas"]
    )

    detail_db_records = crud_detail_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["metadatas"]
    )

    discovery_deleted = len(discovery_db_records["ids"])
    detail_deleted = len(detail_db_records["ids"])

    crud_discovery_db.delete(where={"scheme_id": scheme_id})
    crud_detail_db.delete(where={"scheme_id": scheme_id})

    discovery_remaining = crud_discovery_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["metadatas"]
    )

    detail_remaining = crud_detail_db._collection.get(
        where={"scheme_id": scheme_id},
        include=["metadatas"]
    )

    discovery_after = crud_discovery_db._collection.count()
    detail_after = crud_detail_db._collection.count()

    deletion_verified = (
        len(discovery_remaining["ids"]) == 0
        and len(detail_remaining["ids"]) == 0
    )

    latency = time.perf_counter() - start

    span.update(metadata={
        "scheme_id": scheme_id,
        "discovery_deleted": discovery_deleted,
        "detail_deleted": detail_deleted,
        "total_deleted": discovery_deleted + detail_deleted,
        "deletion_verified": deletion_verified
    })

print("Scheme ID:", scheme_id)

print("\nDiscovery")
print("Before:", discovery_before)
print("Deleted:", discovery_deleted)
print("After:", discovery_after)

print("\nDetail")
print("Before:", detail_before)
print("Deleted:", detail_deleted)
print("After:", detail_after)

print("\nTotal deleted:", discovery_deleted + detail_deleted)
print("Scheme completely removed:", deletion_verified)
print(f"Total deletion latency: {latency:.4f} seconds")

langfuse.flush()

Scheme ID: CRUD_TEST_SCHEME_002

Discovery
Before: 2
Deleted: 2
After: 0

Detail
Before: 3
Deleted: 3
After: 0

Total deleted: 5
Scheme completely removed: True
Total deletion latency: 0.0738 seconds


Failed to export spans batch due to timeout, max retries or shutdown.
